[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vvvvvivekkk/AI-Engineer/blob/main/02-python-for-data/notebooks/01-numpy-basics.ipynb)

# 01 · NumPy basics: arrays, shapes and vectorisation

**What you will learn**
- what a NumPy **array** is (a grid of same-type numbers) and why it is faster than a list
- how to read an array's **shape**, **ndim** and **dtype**
- indexing and slicing in 1-D and 2-D
- **vectorisation** (maths on whole arrays without loops) and how much faster it is
- aggregations with `axis`, boolean masks, broadcasting and seeded random numbers

Run each cell with **Shift + Enter**. Read the text above each cell first.

## 1. Import NumPy

`import numpy as np` loads the library and gives it the short nickname `np`. Everyone uses this nickname, so code you find online will match.

In [1]:
import numpy as np
print("NumPy version:", np.__version__)

NumPy version: 2.5.3


## 2. A list versus an array

A Python **list** repeats itself when you multiply it. A NumPy **array** multiplies every number. Watch the difference.

In [2]:
price_list = [799, 2499, 1199, 249]
price_array = np.array(price_list)

print("list * 2  ->", price_list * 2)
print("array * 2 ->", price_array * 2)

list * 2  -> [799, 2499, 1199, 249, 799, 2499, 1199, 249]
array * 2 -> [1598 4998 2398  498]


## 3. Shape, ndim and dtype

These three facts describe every array:
- **shape**: size along each direction
- **ndim**: number of directions (dimensions)
- **dtype**: the data type of every item

Below, a 2-D array of exam scores: 4 students (rows) × 3 subjects (columns).

In [3]:
scores = np.array([
    [80, 72, 65],   # student 0: maths, science, english
    [65, 90, 70],   # student 1
    [70, 88, 92],   # student 2
    [95, 60, 75],   # student 3
])
print("shape:", scores.shape)
print("ndim: ", scores.ndim)
print("dtype:", scores.dtype)
print("size: ", scores.size, "numbers in total")

shape: (4, 3)
ndim:  2
dtype: int64
size:  12 numbers in total


## 4. Making arrays quickly

You rarely type arrays by hand. These helpers build them:
- `np.arange(start, stop, step)`: like `range`, but returns an array
- `np.linspace(start, stop, n)`: `n` evenly spaced numbers, including both ends
- `np.zeros(shape)` and `np.ones(shape)`: arrays full of 0 or 1

In [4]:
print(np.arange(0, 10, 2))
print(np.linspace(0, 1, 5))
print(np.zeros((2, 3)))
print(np.ones(4, dtype=int))

[0 2 4 6 8]
[0.   0.25 0.5  0.75 1.  ]
[[0. 0. 0.]
 [0. 0. 0.]]
[1 1 1 1]


## 5. Indexing and slicing

Positions start at 0. In 2-D, write `[row, column]`. A colon `:` means "everything along this direction".

In [5]:
print("student 0, all subjects:", scores[0])
print("student 2, science:     ", scores[2, 1])
print("all students, maths:    ", scores[:, 0])
print("students 1-2, first two subjects:\n", scores[1:3, :2])

student 0, all subjects: [80 72 65]
student 2, science:      88
all students, maths:     [80 65 70 95]
students 1-2, first two subjects:
 [[65 90]
 [70 88]]


## 6. Vectorised maths

Arithmetic between arrays of the same shape happens item by item. No loop needed.
Here we compute the bill for four orders: quantity × price × (1 − discount).

In [6]:
quantity = np.array([2, 1, 3, 6])
price = np.array([799, 2499, 1199, 249])
discount = np.array([0.0, 0.10, 0.05, 0.0])

bill = quantity * price * (1 - discount)
print(bill)
print("total:", bill.sum())

[1598.   2249.1  3417.15 1494.  ]
total: 8758.25


## 7. How much faster is vectorisation?

We time the same calculation on one million numbers, first with a Python loop, then with NumPy.
`time.perf_counter()` reads a precise clock. Your exact numbers will differ from your neighbour's; the gap is what matters.

In [7]:
import time

n = 1_000_000
rng = np.random.default_rng(42)
q = rng.integers(1, 6, n)
p = rng.integers(100, 3000, n)

start = time.perf_counter()
loop_result = [q[i] * p[i] for i in range(n)]
loop_seconds = time.perf_counter() - start

start = time.perf_counter()
vec_result = q * p
vec_seconds = time.perf_counter() - start

print(f"loop:       {loop_seconds:.3f} s")
print(f"vectorised: {vec_seconds:.4f} s")
print(f"vectorised is about {loop_seconds / vec_seconds:.0f}x faster")
print("same answer?", np.array_equal(np.array(loop_result), vec_result))

loop:       0.232 s
vectorised: 0.0045 s
vectorised is about 51x faster
same answer? True


## 8. Aggregations and `axis`

- `scores.mean()` averages everything.
- `axis=0` collapses the rows away: one answer **per column** (per subject).
- `axis=1` collapses the columns away: one answer **per row** (per student).

In [8]:
print("overall mean:       ", scores.mean())
print("mean per subject:   ", scores.mean(axis=0))
print("mean per student:   ", scores.mean(axis=1))
print("best score per subj:", scores.max(axis=0))
print("which student is best at each subject:", scores.argmax(axis=0))

overall mean:        76.83333333333333
mean per subject:    [77.5 77.5 75.5]
mean per student:    [72.33333333 75.         83.33333333 76.66666667]
best score per subj: [95 90 92]
which student is best at each subject: [3 1 2]


## 9. Boolean masks

A comparison such as `price > 1000` gives an array of True/False values called a **mask**.
Putting the mask inside square brackets keeps only the True positions.

In [9]:
mask = price > 1000
print("mask:            ", mask)
print("expensive prices:", price[mask])
print("how many?        ", mask.sum())   # True counts as 1, False as 0

mask:             [False  True  True False]
expensive prices: [2499 1199]
how many?         2


## 10. Broadcasting

When shapes differ, NumPy stretches the smaller one if it can. Adding a single number affects every item.
Adding a row of 3 numbers to a 4×3 table adds that row to every row. Here we give +5 bonus marks in science only.

In [10]:
bonus = np.array([0, 5, 0])     # shape (3,) - one value per subject
print(scores + bonus)
print("scores + 1 (a single number is broadcast too):\n", scores + 1)

[[80 77 65]
 [65 95 70]
 [70 93 92]
 [95 65 75]]
scores + 1 (a single number is broadcast too):
 [[81 73 66]
 [66 91 71]
 [71 89 93]
 [96 61 76]]


## 11. Reshape

`reshape` rearranges the same numbers into a new shape. The total count must stay the same (12 numbers can be 3×4, 2×6, 12×1, …).

In [11]:
a = np.arange(12)
print(a)
print(a.reshape(3, 4))
print(a.reshape(2, -1))   # -1 means "work this one out for me" -> (2, 6)

[ 0  1  2  3  4  5  6  7  8  9 10 11]
[[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
[[ 0  1  2  3  4  5]
 [ 6  7  8  9 10 11]]


## 12. Random numbers with a seed

`np.random.default_rng(seed)` makes a random number generator. The same seed gives the same numbers every run, which makes work **reproducible**.
Run this cell twice: the output does not change.

In [12]:
rng = np.random.default_rng(42)
print("5 dice rolls:      ", rng.integers(1, 7, 5))
print("3 decimals in 0-1: ", rng.random(3).round(3))
heights = rng.normal(loc=165, scale=8, size=1000)   # mean 165 cm, spread 8 cm
print(f"1000 heights: mean={heights.mean():.1f}, std={heights.std():.1f}")

5 dice rolls:       [1 5 4 3 3]
3 decimals in 0-1:  [0.697 0.094 0.976]
1000 heights: mean=164.8, std=7.9


## 13. Your turn (mini challenge)

The array below holds weekly study hours for 5 students over 4 weeks (rows = students, columns = weeks).
1. Print the total hours per student.
2. Print the average hours per week (one number per week).
3. Print which students studied more than 30 hours in total.

A worked answer follows in the next cell, but try first!

In [13]:
hours = np.array([
    [5, 7, 6, 8],
    [10, 9, 11, 12],
    [2, 3, 1, 4],
    [8, 8, 9, 7],
    [6, 5, 7, 6],
])
# write your answer here

### One possible answer

In [14]:
total_per_student = hours.sum(axis=1)
print("total per student:", total_per_student)
print("average per week: ", hours.mean(axis=0))
print("students over 30h:", np.where(total_per_student > 30)[0])

total per student: [26 42 10 32 24]
average per week:  [6.2 6.4 6.8 7.4]
students over 30h: [1 3]


## Recap

- Arrays are same-type grids; check `shape`, `ndim`, `dtype` first.
- Use `[row, col]` and `:` to slice.
- Vectorised maths replaces loops and is much faster.
- `axis=0` gives one answer per column; `axis=1` one per row.
- Masks filter; broadcasting stretches; seeds make randomness repeatable.

Next: **02 · pandas loading and cleaning**.